# Central Limit Theorem Demo

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Observe normalized sample means approach a Gaussian distribution.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/clt/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's defaults: means of n Uniform(0, 1) draws, M repetitions, seed 7
# (same parameters; torch's random draws differ from the page's generator)
n, M, seed = 20, 1500, 7
torch.manual_seed(seed)

mu, sigma = 0.5, math.sqrt(1 / 12)              # mean and sd of Uniform(0, 1)
means = torch.rand(M, n, dtype=torch.float64).mean(1)
z = (means - mu) / (sigma / math.sqrt(n))       # standardized sample means

print(f"mean of z = {z.mean().item():+.4f} (0),  variance of z = {z.var().item():.4f} (1)")

# ── Compare quantiles of z with the standard normal ─────────────────────────
normal = torch.distributions.Normal(0.0, 1.0)
print("\n  prob   empirical quantile   N(0,1) quantile")
for q in (0.025, 0.16, 0.5, 0.84, 0.975):
    print(f"  {q:5.3f}      {torch.quantile(z, q).item():+.4f}            {normal.icdf(torch.tensor(q)).item():+.4f}")
inside = (z.abs() < 1.96).double().mean().item()
print(f"\nfraction with |z| < 1.96 = {inside:.4f} (normal: 0.95)")

# ── Skewed input still ends up Gaussian, just more slowly ───────────────────
expo = torch.distributions.Exponential(1.0)
for m in (1, 5, 30, 200):
    zm = (expo.sample((20000, m)).double().mean(1) - 1.0) * math.sqrt(m)
    skew = ((zm - zm.mean()) ** 3).mean() / zm.std() ** 3
    print(f"Exponential, n={m:3d}: skewness of z = {skew.item():.3f}  (theory 2/sqrt(n) = {2 / math.sqrt(m):.3f})")

mean of z = -0.0103 (0),  variance of z = 0.9714 (1)

  prob   empirical quantile   N(0,1) quantile
  0.025      -1.8627            -1.9600
  0.160      -1.0257            -0.9945
  0.500      -0.0114            +0.0000
  0.840      +0.9771            +0.9945
  0.975      +1.9069            +1.9600

fraction with |z| < 1.96 = 0.9553 (normal: 0.95)
Exponential, n=  1: skewness of z = 1.985  (theory 2/sqrt(n) = 2.000)
Exponential, n=  5: skewness of z = 0.908  (theory 2/sqrt(n) = 0.894)
Exponential, n= 30: skewness of z = 0.358  (theory 2/sqrt(n) = 0.365)
Exponential, n=200: skewness of z = 0.159  (theory 2/sqrt(n) = 0.141)
